# 2.1. Data Manipulation

In order to get anything done, we need some way to store and manipulate data. Generally, there are two important things we need to do with data: (i) acquire them; and (ii) process them once they are inside the computer. There is no point in acquiring data without some way to store it, so let us get our hands dirty first by playing with synthetic data. To start, we introduce the -dimensional array, which is also called the tensor.

If you have worked with a Linear Algebra library, then you will find this section familiar. No matter which framework you use, its tensor class (ndarray in MXNet) is similar to a Linear Algebra library with a few killer features. First, GPU is well-supported to accelerate the computation whereas Linear Algebra libraries only supports CPU computation. Second, the tensor class supports automatic differentiation. These properties make the tensor class suitable for deep learning. Throughout the book, when we say tensors, we are referring to instances of the tensor class unless otherwise stated.

## 2.1.1. Getting Started

In this section, we aim to get you up and running, equipping you with the basic math and numerical computing tools that you will build on as you progress through the book. Do not worry if you struggle to grok some of the mathematical concepts or library functions. The following sections will revisit this material in the context of practical examples and it will sink in. On the other hand, if you already have some background and want to go deeper into the mathematical content, just skip this section.
To start, we import the modules from MXNet.

In [1]:
# Todo programa que usted desarrolle debe cargar las siguientes librerías:
use strict;
use warnings;
use Data::Dump qw(dump);
use AI::MXNet qw(mx);

In [2]:
# Para plotear las gráficas, debe cargar las siguientes librerías:
use Chart::Plotly;
use Chart::Plotly::Plot;
use Chart::Plotly::Trace::Scatter;
IPerl->load_plugin('Chart::Plotly'); # Sirve para solo para Jupyter
# Cuando se ejecuta desde la terminal, comentar la carga del plugin y cargar: use Chart::Plotly qw(show_plot);
# En la terminal se utiliza la función show_plot() como muestra el ejemplo en https://metacpan.org/pod/Chart::Plotly

A tensor represents a (possibly multi-dimensional) array of numerical values. With one axis, a tensor is called a vector. With two axes, a tensor is called a matrix. With k > 2 axes, we drop the specialized names and just refer to the object as a kth order tensor.

MXNet provides a variety of functions for creating new tensors prepopulated with values. For example, by invoking arange(n), we can create a vector of evenly spaced values, starting at 0 (included) and ending at n (not included). By default, the interval size is 1. Unless otherwise specified, new tensors are stored in main memory and designated for CPU-based computation.

In [184]:
my $x = mx->nd->arange(start => 0, 
                       stop => 12, 
                       step => 1,
                       ctx => mx->cpu(0));
printf "%s\n", $x->aspdl;
my $y = $x->as_in_context(mx->cpu(1));
print $y;

[0 1 2 3 4 5 6 7 8 9 10 11]
<AI::MXNet::NDArray 12 @cpu(1)>

1

In [185]:
my $z = $x + $y;
print $z->aspdl;

[0 2 4 6 8 10 12 14 16 18 20 22]

1

In [5]:
$y = mx->nd->arange(stop => 12)->reshape([3, 4]);
print $y, $y->aspdl;
$x = $y->transpose();
print $x, $x->aspdl;

<AI::MXNet::NDArray 3x4 @cpu(0)>
[
 [ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
]
<AI::MXNet::NDArray 4x3 @cpu(0)>
[
 [ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]
]


1

In [6]:
$x = mx->nd->arange(stop=>12)->reshape([3,4]);
print $x->aspdl;
$y = mx->nd->arange(stop=>3)->reshape([3,1]);
print $y->aspdl;
$z = $x + $y;
print $z->aspdl;


[
 [ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
]

[
 [0]
 [1]
 [2]
]

[
 [ 0  1  2  3]
 [ 5  6  7  8]
 [10 11 12 13]
]


1

In [7]:
$y = mx->nd->array([2]);
print $y->aspdl;
$z = $x + $y;
print $z->aspdl;

[2]
[
 [ 2  3  4  5]
 [ 6  7  8  9]
 [10 11 12 13]
]


1

In [8]:
$x = mx->nd->arange(stop => 12 * 2) / 2;
print $x->aspdl;

[0 0.5 1 1.5 2 2.5 3 3.5 4 4.5 5 5.5 6 6.5 7 7.5 8 8.5 9 9.5 10 10.5 11 11.5]

1

In [9]:
print $y->dtype;

float32

1

In [10]:
$y = $y->astype('int32');
print $y->dtype;

int32

1

In [11]:
my $z = mx->nd->array([16.9, 8.0, 10.5], dtype=>'int32');
printf "%s\n", $z->dtype;
printf "%s\n", $z->aspdl;

int32
[16 8 10]


1

In [12]:
print $x;

<AI::MXNet::NDArray 24 @cpu(0)>

1

In [13]:
print ref($x);

AI::MXNet::NDArray

1

In [14]:
print dump $x;

bless({
  handle   => bless(do{\(my $o = 94897861659744)}, "NDArrayHandle"),
  writable => 1,
}, "AI::MXNet::NDArray")

1

In [15]:
print $x->handle;

NDArrayHandle=SCALAR(0x564f21551208)

1

We can access a tensor’s shape (the length along each axis) by inspecting its shape property.

In [16]:
$x->shape; # Devuelve una referencia a un arreglo Perl

ARRAY(0x564f21587878)

In [17]:
print dump $x->shape;

[24]

1

In [18]:
print $x->shape->[0];

24

1

In [19]:
my ($rows, $cols) = @{$x->shape};
print dump ($rows, $cols);

(24, undef)

1

In [20]:
print $x->len; # Sinónimo de: $x->shape->[0];

24

1

In [21]:
print $x->ndim;

1

1

If we just want to know the total number of elements in a tensor, i.e., the product of all of the shape elements, we can inspect its size. Because we are dealing with a vector here, the single element of its shape is identical to its size.

In [22]:
print $x->size;

24

1

To change the shape of a tensor without altering either the number of elements or their values, we can invoke the reshape function. For example, we can transform our tensor, x, from a row vector with shape (12,) to a matrix with shape (3, 4). This new tensor contains the exact same values, but views them as a matrix organized as 3 rows and 4 columns. To reiterate, although the shape has changed, the elements have not. Note that the size is unaltered by reshaping.

In [23]:
$x = mx->nd->arange(start => 0, stop => 12)->reshape([3, 4]);
print $x->aspdl;


[
 [ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
]


1

In [24]:
print $x->reshape([-1,2])->aspdl;


[
 [ 0  1]
 [ 2  3]
 [ 4  5]
 [ 6  7]
 [ 8  9]
 [10 11]
]


1

In [25]:
print $x->transpose([0, 1])->aspdl;


[
 [ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
]


1

In [26]:
$x = mx->nd->arange(start => 0, stop => 24);
$x = $x->reshape([2, 3, 4]);
print $x->aspdl;


[
 [
  [ 0  1  2  3]
  [ 4  5  6  7]
  [ 8  9 10 11]
 ]
 [
  [12 13 14 15]
  [16 17 18 19]
  [20 21 22 23]
 ]
]


1

In [27]:
print $x->reshape([$x->len, -1])->aspdl;


[
 [ 0  1  2  3  4  5  6  7  8  9 10 11]
 [12 13 14 15 16 17 18 19 20 21 22 23]
]


1

In [28]:
$x = mx->nd->arange(stop=>24)->reshape([2,3,4]);
print $x->aspdl;


[
 [
  [ 0  1  2  3]
  [ 4  5  6  7]
  [ 8  9 10 11]
 ]
 [
  [12 13 14 15]
  [16 17 18 19]
  [20 21 22 23]
 ]
]


1

In [29]:
print $x->transpose([1,0,2])->aspdl; 


[
 [
  [ 0  1  2  3]
  [12 13 14 15]
 ]
 [
  [ 4  5  6  7]
  [16 17 18 19]
 ]
 [
  [ 8  9 10 11]
  [20 21 22 23]
 ]
]


1

In [30]:
print $x->transpose([1, 2, 0])->aspdl;


[
 [
  [ 0 12]
  [ 1 13]
  [ 2 14]
  [ 3 15]
 ]
 [
  [ 4 16]
  [ 5 17]
  [ 6 18]
  [ 7 19]
 ]
 [
  [ 8 20]
  [ 9 21]
  [10 22]
  [11 23]
 ]
]


1

In [31]:
print $x->transpose()->aspdl;


[
 [
  [ 0 12]
  [ 4 16]
  [ 8 20]
 ]
 [
  [ 1 13]
  [ 5 17]
  [ 9 21]
 ]
 [
  [ 2 14]
  [ 6 18]
  [10 22]
 ]
 [
  [ 3 15]
  [ 7 19]
  [11 23]
 ]
]


1

In [32]:
$y = $x->transpose([1, 2, 0]);
print $y->aspdl;


[
 [
  [ 0 12]
  [ 1 13]
  [ 2 14]
  [ 3 15]
 ]
 [
  [ 4 16]
  [ 5 17]
  [ 6 18]
  [ 7 19]
 ]
 [
  [ 8 20]
  [ 9 21]
  [10 22]
  [11 23]
 ]
]


1

In [33]:
print dump $y->shape;

[3, 4, 2]

1

Reshaping by manually specifying every dimension is unnecessary. If our target shape is a matrix with shape (height, width), then after we know the width, the height is given implicitly. Why should we have to perform the division ourselves? In the example above, to get a matrix with 3 rows, we specified both that it should have 3 rows and 4 columns. Fortunately, tensors can automatically work out one dimension given the rest. We invoke this capability by placing -1 for the dimension that we would like tensors to automatically infer. In our case, instead of calling \\$X->reshape([3, 4]), we could have equivalently called \\$X->reshape([-1, 4]), or \\$X->reshape([3, -1]).

Typically, we will want our matrices initialized either with zeros, ones, some other constants, or numbers randomly sampled from a specific distribution. We can create a tensor representing a tensor with all elements set to 0 and a shape of (2, 3, 4) as follows:

In [34]:
my $var = mx->nd->zeros([2, 3, 4]);

<AI::MXNet::NDArray 2x3x4 @cpu(0)>

In [35]:
print $var->aspdl;


[
 [
  [0 0 0 0]
  [0 0 0 0]
  [0 0 0 0]
 ]
 [
  [0 0 0 0]
  [0 0 0 0]
  [0 0 0 0]
 ]
]


1

Similarly, we can create tensors with each element set to 1 as follows:

In [36]:
$var = mx->nd->ones([2, 3, 4]);

<AI::MXNet::NDArray 2x3x4 @cpu(0)>

In [37]:
print $var->aspdl;


[
 [
  [1 1 1 1]
  [1 1 1 1]
  [1 1 1 1]
 ]
 [
  [1 1 1 1]
  [1 1 1 1]
  [1 1 1 1]
 ]
]


1

In [38]:
$var = mx->nd->full([2, 3, 4], 1e-9);
print $var->aspdl;


[
 [
  [  1e-09   1e-09   1e-09   1e-09]
  [  1e-09   1e-09   1e-09   1e-09]
  [  1e-09   1e-09   1e-09   1e-09]
 ]
 [
  [  1e-09   1e-09   1e-09   1e-09]
  [  1e-09   1e-09   1e-09   1e-09]
  [  1e-09   1e-09   1e-09   1e-09]
 ]
]


1

Often, we want to randomly sample the values for each element in a tensor from some probability distribution. For example, when we construct arrays to serve as parameters in a neural network, we will typically initialize their values randomly. The following snippet creates a tensor with shape (3, 4). Each of its elements is randomly sampled from a standard Gaussian (normal) distribution with a mean of 0 and a standard deviation of 1.

In [39]:
print mx->nd->random_normal(10, 5, shape => [3, 4])->aspdl;


[
 [  21.061    13.87  15.2172  15.9196]
 [ 19.4586  3.82629  1.14486  7.74308]
 [ 12.8969  0.71959 0.115602   8.9599]
]


1

In [40]:
print mx->nd->random->normal(loc => 0, scale => 1, shape => [3, 4])->aspdl;


[
 [  0.244422 -0.0371607   -0.48775 -0.0226173]
 [  0.574614    1.46613    0.68629   0.354961]
 [   1.07317   0.120175   -0.97111  -0.775697]
]


1

We can also specify the exact values for each element in the desired tensor by supplying a Perl array reference containing the numerical values. Here, the outermost array corresponds to axis 0, and the inner list to axis 1.

In [41]:
print mx->nd->array([[2, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])->aspdl;


[
 [2 1 4 3]
 [1 2 3 4]
 [4 3 2 1]
]


1

In [42]:
$x = mx->nd->array([[2, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]]);
print $x->aspdl;


[
 [2 1 4 3]
 [1 2 3 4]
 [4 3 2 1]
]


1

In [43]:
my @array = (1, 2, 3);
print mx->nd->array(\@array)->aspdl;

[1 2 3]

1

In [44]:
my $array2 = [4, 5, 6];
my $nd_arr = mx->nd->array($array2);

<AI::MXNet::NDArray 3 @cpu(0)>

In [45]:
print $nd_arr->aspdl;

[4 5 6]

1

In [46]:
my $arr2 = [[2, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]];

ARRAY(0x564f215a0dd8)

In [47]:
print mx->nd->array($arr2)->aspdl;


[
 [2 1 4 3]
 [1 2 3 4]
 [4 3 2 1]
]


1

In [48]:
my $array2 = $x->asarray;
print dump $array2;

[[2, 1, 4, 3], [1 .. 4], [4, 3, 2, 1]]

1

In [49]:
my @lista2 = @{$x->asarray};
print dump @lista2;

([2, 1, 4, 3], [1 .. 4], [4, 3, 2, 1])

1

## Shared axis slice function

There is a slice function which works with parameters begin, end, [step].
These parameters can be a scalar or an array. In case of scalars, they simply mean the begin, end or step acting on the first axis. Otherwise as an array, the index of their components indicate the respective axis. In this sense, the axis positions is shared among the three parametes. Finally, notice that the last position located in 'end' is not included, just like Python's range.

In [50]:
my $x = mx->nd->arange(stop=>30)->reshape([5, 6]);
$x->aspdl;


[
 [ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]
 [12 13 14 15 16 17]
 [18 19 20 21 22 23]
 [24 25 26 27 28 29]
]


In [51]:
print $x->slice(begin=>[0,], end=>[2,])->aspdl;


[
 [ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]
]


1

In [52]:
print $x->slice(begin=>[0,], end=>[3,], step=>2)->aspdl;


[
 [ 0  1  2  3  4  5]
 [12 13 14 15 16 17]
]


1

In [53]:
print $x->slice(begin=>[0,0], end=>[3,6], step=>[2, 2])->aspdl;


[
 [ 0  2  4]
 [12 14 16]
]


1

In [54]:
print ref($x->slice(begin=>[0,0], end=>[3,6], step=>[2, 2]));

AI::MXNet::NDArray

1

In [55]:
my $y = $x->slice(begin=>[0,0], end=>[3,6], step=>[2, 2]);
print $y->aspdl;


[
 [ 0  2  4]
 [12 14 16]
]


1

In [56]:
# Odd columns
print $x->slice(begin=>[0, 0], end=>$x->shape, step=>[1, 2])->aspdl;


[
 [ 0  2  4]
 [ 6  8 10]
 [12 14 16]
 [18 20 22]
 [24 26 28]
]


1

In [57]:
# Even columns
print $x->slice(begin=>[0, 1], end=>$x->shape, step=>[1, 2])->aspdl;


[
 [ 1  3  5]
 [ 7  9 11]
 [13 15 17]
 [19 21 23]
 [25 27 29]
]


1

In [58]:
# Shared axis slice function does not allow an assignment to change some tensor values.

In [59]:
print $x->slice(begin=>[0,0], end=>[$x->len,2])->aspdl;


[
 [ 0  1]
 [ 6  7]
 [12 13]
 [18 19]
 [24 25]
]


1

In [60]:
my $y = $x->slice(begin=>[0,0], end=>[$x->len,2]);
print $y->aspdl;
print $y->sin()->aspdl;


[
 [ 0  1]
 [ 6  7]
 [12 13]
 [18 19]
 [24 25]
]

[
 [        0  0.841471]
 [-0.279415  0.656987]
 [-0.536573  0.420167]
 [-0.750987  0.149877]
 [-0.905578 -0.132352]
]


1

In [61]:
$x->slice(begin=>[0,0], end=>[$x->len,2]) .= $y->sin();
print $x->aspdl; # The tensor x remains unchanged.


[
 [ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]
 [12 13 14 15 16 17]
 [18 19 20 21 22 23]
 [24 25 26 27 28 29]
]


1

## Full axis slice function

There is a FULL axis slice function which which returns a slice object reference:  AI::MXNet::NDArray::Slice.
This one allows assignments because of the object reference. The syntax does not have begin/end/step as the shared axis. Instead, it represents a whole axis with an $'X'$. If the tensor is unidimensional, the begin and end are just scalars. Otherwise, they must go into brackets. Notice that it also <b>includes</b> the last element.

In [62]:
print ref $x->slice('X', [0, 1]);

AI::MXNet::NDArray::Slice

1

In [63]:
# Each parameter handles full axis: [begin, and] together for each axis.
print $x->slice('X', [0, 1])->aspdl;


[
 [ 0  1]
 [ 6  7]
 [12 13]
 [18 19]
 [24 25]
]


1

In [64]:
# Allows assignment by using .=
$x->slice('X', [0, 1]) .= $y->sin();
print $x->aspdl;


[
 [        0  0.841471         2         3         4         5]
 [-0.279415  0.656987         8         9        10        11]
 [-0.536573  0.420167        14        15        16        17]
 [-0.750987  0.149877        20        21        22        23]
 [-0.905578 -0.132352        26        27        28        29]
]


1

In [196]:
$x = mx->nd->arange(stop=>12)->reshape([3,4]);
print $x->aspdl;


[
 [ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
]


1

In [197]:
$x->slice([0, 1], [2, 3]) .= mx->nd->full([2,2], 10);
print $x->aspdl;


[
 [ 0  1 10 10]
 [ 4  5 10 10]
 [ 8  9 10 11]
]


1

## slice_axis function

When we need a slice of a particular axis, we can use the slice_axis function. It slices along a given axis. It returns an array slice along a given axis starting from the begin index to the end index which is not included. The result is a AI::MXNet::NDArray tensor.

In [66]:
print mx->nd->slice_axis($x, axis=>0, begin=>1, end=>3)->aspdl;


[
 [ 5  6  7  8]
 [ 9 10 11 12]
]


1

In [67]:
print mx->nd->slice_axis($x, axis=>1, begin=>1, end=>4)->aspdl;


[
 [ 2  3  4]
 [ 6  7  8]
 [10 11 12]
]


1

# 2.1.2. Operations

This book is not about software engineering. Our interests are not limited to simply reading and writing data from/to arrays. We want to perform mathematical operations on those arrays. Some of the simplest and most useful operations are the elementwise operations. These apply a standard scalar operation to each element of an array. For functions that take two arrays as inputs, elementwise operations apply some standard binary operator on each pair of corresponding elements from the two arrays. We can create an elementwise function from any function that maps from a scalar to a scalar.

The common standard arithmetic operators (+, -, \*, /, and \*\*) have all been lifted to elementwise operations for any identically-shaped tensors of arbitrary shape. We can call elementwise operations on any two tensors of the same shape. In the following example, we use commas to formulate a 5-element tuple, where each element is the result of an elementwise operation.

# 2.1.2.1. Operations

The common standard arithmetic operators (+, -, \*, /, and **) have all been lifted to elementwise operations.

In [68]:
my $x = mx->nd->array([1, 2, 4, 8]);
my $y = mx->nd->array([2, 2, 2, 2]);

<AI::MXNet::NDArray 4 @cpu(0)>

In [69]:
my $z = $x + $y;
print $z->aspdl;

[3 4 6 10]

1

In [70]:
$z = $x - $y;
print $z->aspdl;

[-1 0 2 6]

1

In [71]:
$z = $x * $y;
print $z->aspdl;

[2 4 8 16]

1

In [72]:
$z = $x / $y;
print $z->aspdl;

[0.5 1 2 4]

1

In [73]:
$z = $x ** $y;# The ** operator is exponentiation
print $z->aspdl;

[1 4 16 64]

1

In [74]:
$y = mx->nd->array([2]);
print "x->shape:", dump $x->shape;
print "\ny->shape:", dump $y->shape;

x->shape:[4]
y->shape:[1]

1

In [75]:
$z = $x + $y; # Bradcast fue posible
print $z->aspdl;

[3 4 6 10]

1

In [76]:
$y = mx->nd->array([2, 2]);
print "x->shape:", dump $x->shape;
print "\ny->shape:", dump $y->shape;

x->shape:[4]
y->shape:[2]

1

In [77]:
#$z = $x + $y; # Bradcast no es posible
#print $z->aspdl;
# MXNetError: Check failed: l == 1 || r == 1: 
# operands could not be broadcast together with shapes [4] [2]

Many more operations can be applied elementwise, including unary operators like exponentiation.

In [78]:
print mx->nd->exp($x)->aspdl;

[2.71828 7.38906 54.5981 2980.96]

1

In [79]:
my $exp_nd = mx->nd->exp($x);

<AI::MXNet::NDArray 4 @cpu(0)>

In addition to elementwise computations, we can also perform linear algebra operations, including vector dot products and matrix multiplication. We will explain the crucial bits of linear algebra (with no assumed prior knowledge) in Section 2.3.

We can also concatenate multiple tensors together, stacking them end-to-end to form a larger tensor. We just need to provide a list of tensors and tell the system along which axis to concatenate. The example below shows what happens when we concatenate two matrices along rows (axis 0, the first element of the shape) vs. columns (axis 1, the second element of the shape). We can see that the first output tensor’s axis-0 length (3+3) is the sum of the two input tensors’ axis-0 lengths (6); while the second output tensor’s axis-1 length (8) is the sum of the two input tensors’ axis-1 lengths (4+4).

In [80]:
$x = mx->nd->arange(stop => 12)->reshape([3, 4]);
print $x->aspdl;


[
 [ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
]


1

In [81]:
$y = mx->nd->array([[2, 1, 4, 3], [1, 2, 3, 4]]);

<AI::MXNet::NDArray 2x4 @cpu(0)>

In [82]:
$z = mx->nd->concat($x, $y, dim => 0);
print $z->aspdl;


[
 [ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [ 2  1  4  3]
 [ 1  2  3  4]
]


1

In [83]:
my @lista2 = ($x, $y, $y, $y);

<AI::MXNet::NDArray 3x4 @cpu(0)><AI::MXNet::NDArray 2x4 @cpu(0)><AI::MXNet::NDArray 2x4 @cpu(0)><AI::MXNet::NDArray 2x4 @cpu(0)>

In [84]:
$z = mx->nd->concat(@lista2, dim => 0);
print $z->aspdl;


[
 [ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [ 2  1  4  3]
 [ 1  2  3  4]
 [ 2  1  4  3]
 [ 1  2  3  4]
 [ 2  1  4  3]
 [ 1  2  3  4]
]


1

In [85]:
print "x->shape:", dump $x->shape;
print "\ny->shape:", dump $y->shape;

x->shape:[3, 4]
y->shape:[2, 4]

1

In [86]:
# $z = mx->nd->concat($x, $y, dim => 1);
# print $z->aspdl;
# MXNetError: Check failed: shape_assign(&(*in_shape)[i], dshape): 
# Incompatible input shape: expected [3,-1], got [2,4]

In [87]:
print $x->aspdl;
print $y->aspdl;


[
 [ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
]

[
 [2 1 4 3]
 [1 2 3 4]
]


1

Observación: Se recomienda utilizar la función concat() en lugar de concatenate() porque la salida de concatenate() hacia una variable no le permite escribir(writable => 0), mientras que concat sí permite la escritura (writable => 1). También se ha encontrado información que concatenate() fue depricado.

Sometimes, we want to construct a binary tensor via logical statements. Take \\$X == \\$Y as an example. For each position, if \\$X and \\$Y are equal at that position, the corresponding entry in the new tensor takes a value of 1, meaning that the logical statement \\$X == \\$Y is true at that position; otherwise that position takes 0.

In [88]:
$x = mx->nd->array([1, 2, 3]);
$y = mx->nd->array([4, 5, 3]);
$z = $x == $y;
print $z->aspdl; # Equivalente a un assert

[0 0 1]

1

Summing all the elements in the tensor yields a tensor with only one element.

In [89]:
$x = mx->nd->arange(stop => 12)->reshape([3, 4]);
print $x->aspdl;
print $x->sum()->aspdl;


[
 [ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
]
[66]

1

In [90]:
print $x->sum(axis => 1)->aspdl;

[6 22 38]

1

In [91]:
# Preserves the dimension of original tensor
print $x->sum(axis => 0, keepdims=>1)->aspdl;


[
 [12 15 18 21]
]


1

In [92]:
print $x->sum(axis => 1)->aspdl;

[6 22 38]

1

In [93]:
print $x;
print $x->aspdl;

<AI::MXNet::NDArray 3x4 @cpu(0)>
[
 [ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
]


1

In [94]:
$y = $x->expand_dims(2);
print $y;
print $y->aspdl;

<AI::MXNet::NDArray 3x4x1 @cpu(0)>
[
 [
  [0]
  [1]
  [2]
  [3]
 ]
 [
  [4]
  [5]
  [6]
  [7]
 ]
 [
  [ 8]
  [ 9]
  [10]
  [11]
 ]
]


1

In [95]:
print $y->squeeze(2);

<AI::MXNet::NDArray 3x4 @cpu(0)>

1

In [96]:
print $x->sum->asscalar, "\n";
print ref($x), "\n";
print ref(\$x->sum()->asscalar);

66
AI::MXNet::NDArray
SCALAR

1

In [97]:
$x = mx->nd->array([2, 3, 4]);
$y = mx->nd->array([1, 5, 2]);
print $x->maximum($y)->aspdl;
#https://mxnet.apache.org/versions/master/api/python/docs/api/np/generated/mxnet.np.maximum.html

[2 5 4]

1

In [98]:
$x = mx->nd->array([-2, -3, 4, 5]);
print $x->maximum(0)->aspdl;

[0 0 4 5]

1

In [99]:
$x = mx->nd->arange(stop=>100) / 10;
print $x->aspdl;

my $trace = [new Chart::Plotly::Trace::Scatter(x => $x->aspdl,
                                               y => $x->maximum(5)->aspdl,
                                               name => 'model',
                                               mode => 'lines')];
                                               
new Chart::Plotly::Plot(traces => $trace, 
                        layout => {title => {text => 'Plot of maximum(5)'},
                                   xaxis => {title => 'X'}, yaxis => {title => 'Y'}});

[0     0.1     0.2     0.3     0.4 0.5     0.6     0.7     0.8     0.9 1     1.1     1.2     1.3     1.4 1.5     1.6     1.7     1.8     1.9 2     2.1     2.2     2.3     2.4 2.5     2.6     2.7     2.8     2.9 3     3.1     3.2     3.3     3.4 3.5     3.6     3.7     3.8     3.9 4     4.1     4.2     4.3     4.4 4.5     4.6     4.7     4.8     4.9 5     5.1     5.2     5.3     5.4 5.5     5.6     5.7     5.8     5.9 6     6.1     6.2     6.3     6.4 6.5     6.6     6.7     6.8     6.9 7     7.1     7.2     7.3     7.4 7.5     7.6     7.7     7.8     7.9 8     8.1     8.2     8.3     8.4 8.5     8.6     8.7     8.8     8.9 9     9.1     9.2     9.3     9.4 9.5     9.6     9.7     9.8     9.9]

In [100]:
my $trace = [new Chart::Plotly::Trace::Scatter(x => $x->aspdl,
                                               y => $x->sin()->aspdl,
                                               name => 'model',
                                               mode => 'lines')];
                                               
new Chart::Plotly::Plot(traces => $trace, 
                        layout => {title => {text => 'Plot of sin(x)'},
                                   xaxis => {title => 'X'}, yaxis => {title => 'Y'}});

In [101]:
my $trace = [new Chart::Plotly::Trace::Scatter(x => $x->aspdl,
                                               y => mx->nd->reverse($x->sin(), axis=>0)->aspdl,
                                               name => 'model',
                                               mode => 'lines')];
                                               
new Chart::Plotly::Plot(traces => $trace, 
                        layout => {title => {text => 'Plot of sin(x)'},
                                   xaxis => {title => 'X'}, yaxis => {title => 'Y'}});

In [102]:
print mx->nd->reverse($x, axis=>0)->aspdl;

[    9.9     9.8     9.7     9.6 9.5     9.4     9.3     9.2     9.1 9     8.9     8.8     8.7     8.6 8.5     8.4     8.3     8.2     8.1 8     7.9     7.8     7.7     7.6 7.5     7.4     7.3     7.2     7.1 7     6.9     6.8     6.7     6.6 6.5     6.4     6.3     6.2     6.1 6     5.9     5.8     5.7     5.6 5.5     5.4     5.3     5.2     5.1 5     4.9     4.8     4.7     4.6 4.5     4.4     4.3     4.2     4.1 4     3.9     3.8     3.7     3.6 3.5     3.4     3.3     3.2     3.1 3     2.9     2.8     2.7     2.6 2.5     2.4     2.3     2.2     2.1 2     1.9     1.8     1.7     1.6 1.5     1.4     1.3     1.2     1.1 1     0.9     0.8     0.7     0.6 0.5     0.4     0.3     0.2     0.1 0]

1

# 2.1.2.2. Additional operations

In [103]:
# Power
$x = mx->nd->arange(stop => 4);
print $x->power(2)->aspdl;
# Instead of mx->nd->power($x, 2);

[0 1 4 9]

1

In [104]:
# Shuffle
$x = mx->nd->arange(stop => 15)->reshape([5, 3]);
print "\nNot shuffled: ", $x->aspdl;
print "\nShuffled:     ", $x->shuffle->aspdl;


Not shuffled: 
[
 [ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]
 [12 13 14]
]

Shuffled:     
[
 [ 0  1  2]
 [ 3  4  5]
 [ 9 10 11]
 [12 13 14]
 [ 6  7  8]
]


1

In [105]:
sub factorial{
  my $n = shift;
  return $n == 0 ? 1 : mx->nd->prod(mx->nd->arange(start => 1, stop => $n + 1))->asscalar;
}

print factorial(5);

120

1

## mx->nd->pick 

Picks elements from an input array according to the input indices along the given axis.

Given an input array of shape (d0, d1) and indices of shape (i0,), the result will be an output array of shape (i0,) with:

https://mxnet.apache.org/versions/1.6/api/r/docs/api/mx.nd.pick.html

In [106]:


$x = mx->nd->arange(stop=>12)->reshape([3,4]);
print $x;
print $x->aspdl;

<AI::MXNet::NDArray 3x4 @cpu(0)>
[
 [ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
]


1

In [107]:
# Considere tomar un elemento de cada fila de la matriz x usando la función pick().
# Debe armar un tensor de índices que contenga un solo índice por columna (eje 1).
printf "[1, 2, 3]: %s\n", mx->nd->pick($x, mx->nd->array([1, 2, 3]), axis => 1)->aspdl;
printf "[3, 3, 3]: %s\n", mx->nd->pick($x, mx->nd->array([3, 3, 3]), axis => 1)->aspdl;
printf "[3, 2, 1]: %s\n", mx->nd->pick($x, mx->nd->array([3, 2, 1]), axis => 1)->aspdl;

[1, 2, 3]: [1 6 11]
[3, 3, 3]: [3 7 11]
[3, 2, 1]: [3 6 9]


1

In [108]:
$x = mx->nd->array([[ 1,  2], [ 3,  4], [ 5,  6]]);
print $x;
print $x->aspdl;

<AI::MXNet::NDArray 3x2 @cpu(0)>
[
 [1 2]
 [3 4]
 [5 6]
]


1

In [109]:
# picks elements with specified indices along axis 0
print mx->nd->pick($x, mx->nd->array([1, 2]), axis => 0)->aspdl;
# = [ 3,  6]

[3 6]

1

In [110]:
# picks elements with specified indices along axis 1
print mx->nd->pick($x, mx->nd->array([1, 0, 1]), axis => 1)->aspdl;
# [2, 3, 6]

[2 3 6]

1

## mx->nd->take

Takes elements from an input array along the given axis.

This function slices the input array along a particular axis with the provided indices.

Given data tensor of rank r >= 1, and indices tensor of rank q, gather entries of the axis dimension of data (by default outer-most one as axis=0) indexed by indices, and concatenates them in an output tensor of rank q + (r - 1).
https://mxnet.apache.org/versions/1.6/api/r/docs/api/mx.nd.take.html

In [111]:
print $x->aspdl;
print mx->nd->take($x, mx->nd->array([2, 0, 1, 1]), axis => 0)->aspdl;


[
 [1 2]
 [3 4]
 [5 6]
]

[
 [5 6]
 [1 2]
 [3 4]
 [3 4]
]


1

In [112]:
print $x->aspdl;
print mx->nd->take($x, mx->nd->array([1, 0]), axis => 1)->aspdl;


[
 [1 2]
 [3 4]
 [5 6]
]

[
 [2 1]
 [4 3]
 [6 5]
]


1

## mx->nd->contrib->arange_like

Return an array with evenly spaced values. If axis is not given, the output will have the same shape as the input array. Otherwise, the output will be a 1-D array with size of the specified axis in input shape.

In [113]:
$y = mx->nd->array([[0.14883883, 0.7772398,  0.94865847, 0.7225052 ],
                    [0.23729339, 0.6112595,  0.66538996, 0.5132841 ],
                    [0.30822644, 0.9912457,  0.15502319, 0.7043658 ]]);
     
print mx->nd->contrib->arange_like($y)->aspdl;


[
 [ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
]


1

In [114]:
printf "Shape as NDArray: %s\n", ref $x->shape_array();
printf "Shape: %s\n", $x->shape_array()->aspdl;

Shape as NDArray: AI::MXNet::NDArray
Shape: [3 2]


1

In [115]:
printf "Shape as Perl ARRAY: %s\n", ref $x->shape;
printf "Shape: %s\n", dump $x->shape; # [3 4]

Shape as Perl ARRAY: ARRAY
Shape: [3, 2]


1

## mx->nd->flatten

Flattens the input array into a 2-D array by collapsing the higher dimensions.

For an input array with shape (d1, d2, ..., dk), flatten operation reshapes the input array into an output array of shape (d1, d2*...*dk). Note that the behavior of this function is different from numpy.ndarray.flatten, which behaves similar to mxnet.ndarray.reshape((-1,)).

https://mxnet.apache.org/versions/1.6/api/r/docs/api/mx.nd.flatten.html

In [116]:
$x = mx->nd->array([[ [1,2,3], [4,5,6], [7,8,9]], [[1,2,3], [4,5,6], [7,8,9] ]]);

<AI::MXNet::NDArray 2x3x3 @cpu(0)>

In [117]:
mx->nd->flatten($x)->aspdl;
# [[ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9.],
# [ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9.]]


[
 [1 2 3 4 5 6 7 8 9]
 [1 2 3 4 5 6 7 8 9]
]


In [118]:
mx->nd->flatten(mx->nd->array([[1, 2]]))->aspdl;


[
 [1 2]
]


In [119]:
$x = mx->nd->array([[
[1,2,3],
[4,5,6],
[7,8,9]
],
[    [1,2,3],
[4,5,6],
[7,8,9]
]]);

<AI::MXNet::NDArray 2x3x3 @cpu(0)>

In [120]:
# Flattens the input array into a 2-D array by collapsing the higher dimensions.
my $y = mx->nd->flatten($x);
# Note that the behavior of this function is different from numpy.ndarray.flatten, 
# which behaves similar to mxnet.ndarray.reshape((-1,)).
print $y->aspdl;


[
 [1 2 3 4 5 6 7 8 9]
 [1 2 3 4 5 6 7 8 9]
]


1

## full()

In [121]:
mx->nd->full([2, 2], 5)->aspdl;


[
 [5 5]
 [5 5]
]


In [122]:
$var = mx->nd->full([2, 3, 4], 10);
print $var->aspdl;


[
 [
  [10 10 10 10]
  [10 10 10 10]
  [10 10 10 10]
 ]
 [
  [10 10 10 10]
  [10 10 10 10]
  [10 10 10 10]
 ]
]


1

# mx->nd->swapaxes
https://mxnet.apache.org/versions/master/api/python/docs/api/np/generated/mxnet.np.swapaxes.html

swapaxes(a, axis1, axis2)
Interchange two axes of an array.

Parameters
a (ndarray) – Input array.

axis1 (int) – First axis.

axis2 (int) – Second axis.

Returns
a_swapped – Swapped array. This is always a copy of the input array.

Return type
ndarray

In [123]:
$x = mx->nd->array([[1, 2, 3]]);
print mx->nd->swapaxes($x, 0, 1)->aspdl;
# array([[1.],
#        [2.],
#        [3.]])


[
 [1]
 [2]
 [3]
]


1

In [124]:
$x = mx->nd->array([[[0,1], [2,3]], [[4,5], [6,7]]]);
print mx->nd->swapaxes($x, 0, 2)->aspdl;
# array([[[0., 4.],
#         [2., 6.]],
#        [[1., 5.],
#        [3., 7.]]])


[
 [
  [0 4]
  [2 6]
 ]
 [
  [1 5]
  [3 7]
 ]
]


1

# 2.1.3. Broadcasting Mechanism

In the above section, we saw how to perform elementwise operations on two tensors of the same shape. Under certain conditions, even when shapes differ, we can still perform elementwise operations by invoking the broadcasting mechanism. This mechanism works in the following way: First, expand one or both arrays by copying elements appropriately so that after this transformation, the two tensors have the same shape. Second, carry out the elementwise operations on the resulting arrays.

In most cases, we broadcast along an axis where an array initially only has length 1, such as in the following example:

In [125]:
my $a = mx->nd->arange(stop => 3)->reshape([3, 1]);
my $b = mx->nd->arange(stop => 2)->reshape([1, 2]);
print $a->aspdl;
print $b->aspdl;


[
 [0]
 [1]
 [2]
]

[
 [0 1]
]


1

Since a and b are 3 x 1 and 1 x 2 matrices respectively, their shapes do not match up if we want to add them. We broadcast the entries of both matrices into a larger 3 x 2 matrix as follows: for matrix a it replicates the columns and for matrix b it replicates the rows before adding up both elementwise.

In [126]:
print (($a + $b)->aspdl);


[
 [0 1]
 [1 2]
 [2 3]
]


1

## 2.1.5. Saving Memory

Running operations can cause new memory to be allocated to host results. For example, if we write \\$Y = \\$X + \\$Y, we will dereference the tensor that \\$Y used to point to and instead point \\$Y at the newly allocated memory. In the following example, we demonstrate this with Perl’s handle attribute, which gives us the exact address of the referenced object in memory. After running \\$Y = \\$Y + \\$X, we will find that handle(\\$Y) points to a different location. That is because Perl first evaluates \\$Y + \\$X, allocating new memory for the result and then makes \\$Y point to this new location in memory.

In [127]:
$x = mx->nd->arange(stop => 9)->reshape([3, 3]);
my $y = mx->nd->arange(stop => 9)->reshape([3, 3]);
print dump 'handle(Y) before:', $y->handle;

(
  "handle(Y) before:",
  bless(do{\(my $o = 94897862158576)}, "NDArrayHandle"),
)

1

In [128]:
$y = $y + $x;

<AI::MXNet::NDArray 3x3 @cpu(0)>

In [129]:
print dump 'handle(Y) after:', $y->handle;

(
  "handle(Y) after:",
  bless(do{\(my $o = 94897862318336)}, "NDArrayHandle"),
)

1

This might be undesirable for two reasons. First, we do not want to run around allocating memory unnecessarily all the time. In machine learning, we might have hundreds of megabytes of parameters and update all of them multiple times per second. Typically, we will want to perform these updates in place. Second, we might point at the same parameters from multiple variables. If we do not update in place, other references will still point to the old memory location, making it possible for parts of our code to inadvertently reference stale parameters.

Fortunately, performing in-place operations is easy. We can assign the result of an operation to a previously allocated array with slice notation, e.g., \\$Y .= <expression>. To illustrate this concept, we first create a new matrix \\$Z with the same shape as another \\$Y, using zeros_like to allocate a block of 0 entries.

In [130]:
my $z = mx->nd->zeros_like($y);
print dump 'handle(Z) before:', $z->handle;

(
  "handle(Z) before:",
  bless(do{\(my $o = 94897862317328)}, "NDArrayHandle"),
)

1

In [131]:
$z .= $y + $x;
print dump 'handle(Z) after:',  $z->handle;

(
  "handle(Z) after:",
  bless(do{\(my $o = 94897862317328)}, "NDArrayHandle"),
)

1

If the value of \\$X is not reused in subsequent computations, we can also use \\$X .= \\$X + \\$Y or \\$X += \\$Y to reduce the memory overhead of the operation.

In [132]:
print dump 'handle(X) before:', $x->handle;
$x += $y;
print dump 'handle(X) after:',  $x->handle;

(
  "handle(X) before:",
  bless(do{\(my $o = 94897862330560)}, "NDArrayHandle"),
)(
  "handle(X) after:",
  bless(do{\(my $o = 94897862330560)}, "NDArrayHandle"),
)

1

If you need to save MXNet tensors as a file, you can use the following functions.

In [133]:
# mx->nd->save($x, 'temp.mat'); # https://mxnet.apache.org/versions/1.6/api/r/docs/api/mx.nd.save.html
mx->nd->save('temp.ndarray', [$x]);

In [134]:
# And then, load from that file.
# $x = mx->nd->load('temp.ndarray'); #https://mxnet.apache.org/versions/1.6/api/r/docs/api/mx.nd.load.html
$y = mx->nd->load('temp.ndarray');
print $y->[0]->aspdl;


[
 [ 0  3  6]
 [ 9 12 15]
 [18 21 24]
]


1

# mx->nd->argsort

Returns the indices that would sort an input array along the given axis.

This function performs sorting along the given axis and returns an array of indices having same shape as an input array that index data in sorted order.

https://mxnet.apache.org/versions/1.6/api/r/docs/api/mx.nd.argsort.html

In [135]:
# take_along_axis
$x = mx->nd->array([[[2, 3, 4]], [[ 1, 0, 0]]]);
my $index_array = mx->nd->argsort($x, axis => 0);
# mx->nd->take_along_axis($x, $index_array, axis => 0);
# array([[[1., 0., 0.]],
#        [[2., 3., 4.]]])

<AI::MXNet::NDArray 2x1x3 @cpu(0)>

In [136]:
$x = mx->nd->array([ 0.156,  0.3677,  0.2776]);
# sort along axis -1
print mx->nd->argsort($x)->aspdl;
# [[ 1.,  0.,  2.],
# [ 0.,  2.,  1.]]

[0 2 1]

1

In [137]:
print $x->aspdl;

[  0.156  0.3677  0.2776]

1

In [138]:
print dump [map {sprintf '%.2f', $_} @{$x->asarray}];

[0.16, 0.37, 0.28]

1

In [139]:
$x = mx->nd->array([ 5,  1, 4,  3]);
print mx->nd->sort($x)->aspdl;

[1 3 4 5]

1

In [140]:
# sort along axis 0
print mx->nd->argsort($x, axis => 0)->aspdl;
# [[ 1.,  0.,  1.]
# [ 0.,  1.,  0.]]

[1 3 2 0]

1

In [141]:
# flatten and then sort
print mx->nd->argsort($x, axis => 'None')->aspdl;
# [ 3.,  1.,  5.,  0.,  4.,  2.]

[1 3 2 0]

1

In [142]:
# flatten and then sort
print mx->nd->argsort($x, axis => 'None', is_ascend => 0)->aspdl;
# [2 0 4 1 5 3]

[0 2 3 1]

1

# mx->nd->linalg->det
Compute the determinant of an array.

Parameters
a ((.., M, M) ndarray) – Input array to compute determinants for.

Returns
det – Determinant of a.

Return type
(..) ndarray

https://mxnet.apache.org/versions/master/api/python/docs/api/np/generated/mxnet.np.linalg.det.html#mxnet.np.linalg.det

In [143]:
$x = mx->nd->array([[1, 2], [3, 4]]);
print mx->nd->linalg->det($x)->aspdl;
# -2.0

[-2]

1

In [144]:
$x = mx->nd->array([ [[1, 2], [3, 4]], [[1, 2], [2, 1]], [[1, 3], [3, 1]] ]);
print mx->nd->linalg->det($x)->aspdl;
#array([-2., -3., -8.])

[-2 -3 -8]

1

In [145]:
$x = mx->nd->array([ [[[1, 2], [3, 4]]], [[[1, 2], [2, 1]]], [[[1, 3], [3, 1]]] ]);
print mx->nd->linalg->det($x)->aspdl;
#array([[-2.],
#       [-3.],
#       [-8.]])


[
 [-2]
 [-3]
 [-8]
]


1

# mx->nd->linalg->inverse

Compute the (multiplicative) inverse of a matrix.

Given a square matrix a, return the matrix ainv satisfying dot(a, ainv) = dot(ainv, a) = eye(a.shape[0]).

Parameters
a ((.., M, M) ndarray) – Matrix to be inverted.

Returns
ainv – (Multiplicative) inverse of the matrix a.

Return type
(.., M, M) ndarray

Raises
MXNetError – If a is not square or inversion fails.

https://mxnet.apache.org/versions/master/api/python/docs/api/np/generated/mxnet.np.linalg.inv.html#mxnet.np.linalg.inv

In [146]:
$x = mx->nd->array([[1, 2], [3, 4]]);
print mx->nd->linalg->inverse($x)->aspdl;
# [[  -2.00000,    1.00000 ],
#  [   1.50000,   -0.50000 ]]


[
 [  -2    1]
 [ 1.5 -0.5]
]


1

In [147]:
print mx->nd->dot($x, mx->nd->linalg->inverse($x))->aspdl;
#[[   1.00000,    0.00000 ],
# [   0.00000,    1.00000 ]]


[
 [1 0]
 [0 1]
]


1

In [148]:
$x = mx->nd->array([[1, 3], [3, 5]]);
print mx->nd->linalg->inverse($x)->aspdl;
# [[  -1.25000,    0.75000 ],
#  [   0.75000,   -0.25000 ]]


[
 [  -1.25    0.75]
 [   0.75   -0.25]
]


1

In [149]:
$x = mx->nd->array([[[1., 2.], [3., 4.]], [[1, 3], [3, 5]]]);
print mx->nd->linalg->inverse($x)->aspdl;
#  array([[[-2.        ,  1.        ],
#          [ 1.5       , -0.5       ]],
#         [[-1.2500001 , 0.75000006],
#          [ 0.75000006, -0.25000003]]])


[
 [
  [  -2    1]
  [ 1.5 -0.5]
 ]
 [
  [  -1.25    0.75]
  [   0.75   -0.25]
 ]
]


1

##  mx->nd->sample_multinomial or mx->nd->random->multinomial

Concurrent sampling from multiple multinomial distributions.

data is an n dimensional array whose last dimension has length k, where k is the number of possible outcomes of each multinomial distribution. This operator will draw shape samples from each distribution. If shape is empty one sample will be drawn from each distribution.

If get_prob is true, a second array containing log likelihood of the drawn samples will also be returned. This is usually used for reinforcement learning where you can provide reward as head gradient for this array to estimate gradient.

Note that the input distribution must be normalized, i.e. data must sum to 1 along its last axis.

https://mxnet.apache.org/versions/1.6/api/r/docs/api/mx.nd.sample.multinomial.html

In [150]:
my $probs = mx->nd->array([[0, 0.1, 0.2, 0.3, 0.4], [0.4, 0.3, 0.2, 0.1, 0]]);

# Draw a single sample for each distribution
mx->random->seed(1);
print mx->nd->sample_multinomial($probs)->aspdl; # [3, 0]

[3 0]

1

In [151]:
# // Draw a vector containing two samples for each distribution
mx->random->seed(1);
print mx->nd->sample_multinomial($probs, shape=>2)->aspdl; # [[3, 1], [0, 1]]


[
 [3 1]
 [0 1]
]


1

In [152]:
# // Draw a vector containing two samples for each distribution
mx->random->seed(1);
print mx->nd->random->multinomial($probs, shape=>2)->aspdl; # [[3, 1], [0, 1]]


[
 [3 1]
 [0 1]
]


1

In [153]:
# requests log likelihood
mx->random->seed(1);
my $vals_y_likelihood = mx->nd->sample_multinomial($probs, get_prob=>1); # [2, 1], [0.2, 0.3]
printf "%s\n", $vals_y_likelihood;
print "@$vals_y_likelihood\n";
printf "values: %s\n", $vals_y_likelihood->[0]->aspdl;
printf "likelihood: %s\n", $vals_y_likelihood->[1]->aspdl;

ARRAY(0x564f2160f398)
<AI::MXNet::NDArray 2 @cpu(0)> <AI::MXNet::NDArray 2 @cpu(0)>
values: [3 0]
likelihood: [-1.20397 -0.916291]


1

## mx->nd->random->randn() 

Draw random samples from a normal (Gaussian) distribution.

Samples are distributed according to a normal distribution parametrized by loc (mean) and scale (standard deviation).

Parameters
loc (float or NDArray) – Mean (centre) of the distribution.

scale (float or NDArray) – Standard deviation (spread or width) of the distribution.

shape (int or tuple of ints) – The number of samples to draw. If shape is, e.g., (m, n) and loc and scale are scalars, output shape will be (m, n). If loc and scale are NDArrays with shape, e.g., (x, y), then output will have shape (x, y, m, n), where m*n samples are drawn for each [loc, scale) pair.

dtype ({'float16', 'float32', 'float64'}) – Data type of output samples. Default is ‘float32’

ctx (Context) – Device context of output. Default is current context. Overridden by loc.context when loc is an NDArray.

out (NDArray) – Store output to an existing NDArray.

Returns
If input shape has shape, e.g., (m, n) and loc and scale are scalars, output shape will be (m, n). If loc and scale are NDArrays with shape, e.g., (x, y), then output will have shape (x, y, m, n), where m*n samples are drawn for each [loc, scale) pair.

Return type
NDArray

In [154]:
mx->random->seed(1);
$x = mx->nd->random->randn(5, 4);
print $x->aspdl;


[
 [-0.677652  0.100739  0.575954 -0.346925]
 [-0.221343  -1.80472 -0.806429   1.22033]
 [  2.23236  0.200702 -0.549686  -0.19819]
 [-0.385779   1.37109   -0.2379   0.14868]
 [-0.498513 -0.848158 0.0781115 -0.382413]
]


1

In [155]:
mx->random->seed(1);
print mx->nd->random->randn(2, 3, loc=>5, scale=>1)->aspdl;
# [[4.32235 5.10074 5.57595]
# [4.65307 4.77866 3.19528]]


[
 [4.32235 5.10074 5.57595]
 [4.65307 4.77866 3.19528]
]


1

## mx->nd->random->uniform()

(low=0, high=1, shape=_Null, dtype=_Null, ctx=None, out=None, \**kwargs)

Draw random samples from a uniform distribution.

Samples are uniformly distributed over the half-open interval [low, high) (includes low, but excludes high).

Parameters
low (float or NDArray, optional) – Lower boundary of the output interval. All values generated will be greater than or equal to low. The default value is 0.

high (float or NDArray, optional) – Upper boundary of the output interval. All values generated will be less than high. The default value is 1.0.

shape (int or tuple of ints, optional) – The number of samples to draw. If shape is, e.g., (m, n) and low and high are scalars, output shape will be (m, n). If low and high are NDArrays with shape, e.g., (x, y), then output will have shape (x, y, m, n), where m*n samples are drawn for each [low, high) pair.

dtype ({'float16', 'float32', 'float64'}, optional) – Data type of output samples. Default is ‘float32’

ctx (Context, optional) – Device context of output. Default is current context. Overridden by low.context when low is an NDArray.

out (NDArray, optional) – Store output to an existing NDArray.

Returns
An NDArray of type dtype. If input shape has shape, e.g., (m, n) and low and high are scalars, output shape will be (m, n). If low and high are NDArrays with shape, e.g., (x, y), then the return NDArray will have shape (x, y, m, n), where m*n uniformly distributed samples are drawn for each [low, high) pair.

Return type
NDArray

In [156]:
mx->random->seed(1);
print mx->nd->random->uniform(0, 1)->aspdl;

[0.523833]

1

In [157]:
mx->random->seed(1);
print mx->nd->random->uniform(0, 1, ctx=>mx->cpu(1));

<AI::MXNet::NDArray 1 @cpu(1)>

1

In [158]:
mx->random->seed(1);
print mx->nd->random->uniform(-1, 1, shape=>[2,])->aspdl;
# [0.0476667 -0.889973]

[0.0476667 -0.889973]

1

In [159]:
mx->random->seed(1);
my $low  = mx->nd->array([1,2,3]);
my $high = mx->nd->array([2,3,4]);
print mx->nd->random->uniform($low, $high, shape=>2)->aspdl;
# [[1.52383 1.05501]
#  [2.03996 2.59453]
#  [3.18597 3.69035]]


[
 [1.52383 1.05501]
 [2.03996 2.59453]
 [3.18597 3.69035]
]


1

## mx->nd->randint()

(low, high, shape=_Null, dtype=_Null, ctx=None, out=None, \**kwargs)[source]

Draw random samples from a discrete uniform distribution.

Samples are uniformly distributed over the half-open interval [low, high) (includes low, but excludes high).

Parameters
low (int, required) – Lower boundary of the output interval. All values generated will be greater than or equal to low.

high (int, required) – Upper boundary of the output interval. All values generated will be less than high.

shape (int or tuple of ints, optional) – The number of samples to draw. If shape is, e.g., (m, n) and low and high are scalars, output shape will be (m, n).

dtype ({'int32', 'int64'}, optional) – Data type of output samples. Default is ‘int32’

ctx (Context, optional) – Device context of output. Default is current context. Overridden by low.context when low is an NDArray.

out (NDArray, optional) – Store output to an existing NDArray.

Returns
An NDArray of type dtype. If input shape has shape, e.g., (m, n), the returned NDArray will shape will be (m, n). Contents of the returned NDArray will be samples from the interval [low, high).

Return type
NDArray

https://mxnet.apache.org/versions/1.6/api/r/docs/api/mx.nd.random.randint.html

In [160]:
mx->random->seed(1);
print mx->nd->random->randint(5, 100)->aspdl;
# [61]

[61]

1

In [161]:
mx->random->seed(1);
print mx->nd->random->randint(-10, 2, ctx=>mx->cpu(1))->aspdl;
# [ -2]

[-2]

1

In [162]:
mx->random->seed(1);
print mx->nd->random->randint(-10, 10, shape=>[2,])->aspdl;
# [6 -2]

[6 -2]

1

## 2.1.6. Conversion to Other Perl Objects

Converting to a tensor (NDarray), or vice versa, is easy. The converted result does not share memory. This minor inconvenience is actually quite important: when you perform operations on the CPU or on GPUs, you do not want to halt computation, waiting to see whether the Perl array might want to be doing something else with the same chunk of memory.

In [163]:
my $A = [1, 2, 3]; # Perl array reference
my @B = (4, 5, 6); # Perl array list
print ref ($A);

ARRAY

1

In [164]:
my $C = mx->nd->array($A); # Obtain a NDArray out of a Perl array reference
print ref($C), "\n";
print $C->aspdl;

AI::MXNet::NDArray
[1 2 3]

1

In [165]:
my $D = mx->nd->array(\@B, dtype => 'int32'); # Obtain a NDArray out of a Perl array list
print ref($D), "\n";
print $D->aspdl;
print "\n", $D->dtype;

AI::MXNet::NDArray
[4 5 6]
int32

1

In [166]:
my $E = $C->asarray; # Obtain a Perl array reference out of a NDArray
print ref($E), "\n";
print dump $E;

ARRAY
[1, 2, 3]

1

In [167]:
my @F = @{$C->asarray}; # Obtain a Perl array list out of NDArray
print dump @F;

(1, 2, 3)

1

### Other types of conversions

To convert a size-1 tensor to a Perl scalar, we can invoke the ->asscalar attribute.

In [168]:
my $a = mx->nd->array([3.5]);

<AI::MXNet::NDArray 1 @cpu(0)>

In [169]:
print $a, "\n", $a->dtype, "\n", $a->asscalar;

<AI::MXNet::NDArray 1 @cpu(0)>
float32
3.5

1

We can see 3 mini-batches of data (and labels), the first ones with 3 samples, whereas the last one with 1 sample, which makes sense given we started with a dataset of 10 samples. When comparing the shape of the batches to the samples returned by the Dataset, we’ve gained an extra dimension at the start which is sometimes called the batch axis.

Our data_loader loop will stop when every sample of dataset has been returned as part of a batch. Sometimes the dataset length isn’t divisible by the mini-batch size, leaving a final batch with a smaller number of samples. 

DataLoader’s default behavior is `keep` to return this smaller mini-batch, but this can be changed by setting the last_batch parameter to `discard` (which ignores the last batch) or `rollover` (which starts the next epoch with the remaining samples).

## 2.1.10. Complementary functions for NDArray manipulation

In [170]:
use AI::MXNet::Base; # Available functions product enumerate assert zip check_call build_param_doc pdl cat dog svd bisect_left pdl_shuffle as_array ascsr rand_sparse
# https://metacpan.org/pod/AI::MXNet::Base

In [171]:
my $x = mx->nd->arange(stop=>5);
my $y = $x + 5;
print $x->aspdl, " ", $y->aspdl, "\n";

[0 1 2 3 4] [5 6 7 8 9]


1

In [172]:
for (zip($x, $y)) {
  my ($x1, $y1) = @$_;
  print $x1->aspdl, " ", $y1->aspdl, "\n";
}

[0] [5]
[1] [6]
[2] [7]
[3] [8]
[4] [9]


In [173]:
printf "%s %s\n", $_->[0]->aspdl, $_->[1]->aspdl for (zip($x, $y));

[0] [5]
[1] [6]
[2] [7]
[3] [8]
[4] [9]


In [174]:
print map {$_->[0]->aspdl, " ", $_->[1]->aspdl, "\n";} zip($x, $y)

[0] [5]
[1] [6]
[2] [7]
[3] [8]
[4] [9]


1

In [175]:
for (enumerate($x)){
  my ($i, $x1) = @$_;
  printf "%d %s\n", $i, $x1->aspdl;
}

0 [0]
1 [1]
2 [2]
3 [3]
4 [4]


In [176]:
printf "%d %s\n", $_->[0], $_->[1]->aspdl for (enumerate($x));

0 [0]
1 [1]
2 [2]
3 [3]
4 [4]


In [177]:
print map { sprintf("%d %s\n", $_->[0], $_->[1]->aspdl) } enumerate($x);

0 [0]
1 [1]
2 [2]
3 [3]
4 [4]


1

## 2.1.7. Gluon Datasets and DataLoader

Dataset objects are used to represent collections of data, and include methods to load and parse the data (that is often stored on disk). Gluon has a number of different Dataset classes for working with image data straight out-of-the-box, but we’ll use the ArrayDataset to introduce the idea of a Dataset.

We first start by generating random data X (with 2 variables) and corresponding random labels y to simulate a typical supervised learning task. We generate 10 samples and we pass them all to the ArrayDataset.

In [181]:
my $X = mx->nd->arange(stop=>20)->reshape([10, 2]);
mx->random->seed(1);
my $y = mx->nd->random->randint(0, 2, shape=>[10, 1]);
my $dataset = mx->gluon->data->ArrayDataset(data => $X, label => $y);

printf "Original X for training: %s\n", $X->aspdl;
printf "Original y training: %s\n", $y->aspdl;

my $train_data = mx->gluon->data->DataLoader($dataset, batch_size => 3, shuffle => 1, last_batch => 'rollover'); # keep|discard|rollover

# The number of batches can be obtained by its property len().
print "num_batches: ", $train_data->len();

Original X for training: 
[
 [ 0  1]
 [ 2  3]
 [ 4  5]
 [ 6  7]
 [ 8  9]
 [10 11]
 [12 13]
 [14 15]
 [16 17]
 [18 19]
]

Original y training: 
[
 [0]
 [0]
 [1]
 [1]
 [0]
 [1]
 [1]
 [1]
 [0]
 [1]
]

num_batches: 3

1

In [179]:
# Minibatches of the training dataset
for (my ($i, $batch, $X, $y) = 0; eval{ $batch = <$train_data>, ($X, $y) = @$batch}; $i++){
  print "X:", $X->aspdl;
  print "y:", $y->aspdl, "\n";
}

X:
[
 [16 17]
 [18 19]
 [ 2  3]
]
y:
[
 [0]
 [1]
 [0]
]

X:
[
 [12 13]
 [ 0  1]
 [14 15]
]
y:
[
 [1]
 [0]
 [1]
]

X:
[
 [4 5]
 [8 9]
 [6 7]
]
y:
[
 [1]
 [0]
 [1]
]



In [180]:
my $X = mx->nd->arange(start=>20, stop=>30)->reshape([5, 2]);
mx->random->seed(2);
my $y = mx->nd->random->randint(0, 2, shape=>[5, 1]);
my $dataset = mx->gluon->data->ArrayDataset(data => $X, label => $y);

printf "Original X for testing: %s\n", $X->aspdl;
printf "Original y testing: %s\n", $y->aspdl;

my $test_data = mx->gluon->data->DataLoader($dataset, batch_size => 3, shuffle => 0, last_batch => 'keep'); # keep|discard|rollover

# The number of batches can be obtained by its property len().
printf "num_batches: %d\n\n", $test_data->len();

Original X for testing: 
[
 [20 21]
 [22 23]
 [24 25]
 [26 27]
 [28 29]
]

Original y testing: 
[
 [1]
 [0]
 [1]
 [1]
 [1]
]

num_batches: 2



1

In [181]:
# Minibatches of the test dataset

for (my ($i, $batch, $X, $y) = 0; eval{ $batch = <$test_data>, ($X, $y) = @$batch}; $i++){
  printf "X[%d]: %s\n", $i, $X->aspdl;
  printf "y[%d]: %s\n", $i, $y->aspdl;
}

X[0]: 
[
 [20 21]
 [22 23]
 [24 25]
]

y[0]: 
[
 [1]
 [0]
 [1]
]

X[1]: 
[
 [26 27]
 [28 29]
]

y[1]: 
[
 [1]
 [1]
]



## 2.1.9. Summary

* The main interface to store and manipulate data for deep learning is the tensor (n-dimensional array). It provides a variety of functionalities including basic mathematics operations, broadcasting, indexing, slicing, memory saving, and conversion to other Perl objects.